In [ ]:
# Imports
import numpy as np
import pandas as pd
import json
from pathlib import Path
from collections import Counter
import os
import csv
from datetime import datetime

# Sklearn imports for data handling and metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, classification_report, precision_recall_curve)

# PyTorch imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# For progress bars
from tqdm import tqdm

# For plotting
import matplotlib.pyplot as plt
import seaborn as sns

# For display in notebooks
from IPython.display import display

# Utility functions
from utils import (load_and_extract_entities, encode_and_pad, train_epoch, evaluate, 
                   get_predictions, plot_training_curves, find_optimal_threshold, evaluate_model)

# Model architectures
from sequence_models import get_model, MODELS

Config

In [ ]:
# Data and horizon (must match pre-processing output)
DATA = "nph"   # 'pituitary' | 'nph'
HORIZON = 365  # None (baseline) | 30 | 90 | 365 | "firstcontact"

# Model selection (must match a key in MODELS from sequence_models.py)
MODEL_NAME = "gru_bi"  # e.g. 'lstm', 'gru' etc.

# Generic config
RANDOM_STATE = 42
SCORING_METRIC = "roc_auc"

print("DATA:", DATA, "| HORIZON:", HORIZON, "| MODEL:", MODEL_NAME)

In [ ]:
# Fixed hyperparameters (rarely changed)
EMBED_DIM = 100
HIDDEN_DIM = 128
OUTPUT_DIM = 1
BATCH_SIZE = 32
DROPOUT = 0.5
NUM_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 0.001
LEARNING_RATE = 0.001

# Search configuration
USE_HP_SEARCH = False
LEARNING_RATES = [0.0005, 0.001, 0.002]
HIDDEN_DIMS = [64, 128, 256]
DROPOUT_RATES = [0.3, 0.5, 0.7]
NUM_EPOCHS_SEARCH = 10

# Effective hyperparameters for this run (config default; updated after HP search if USE_HP_SEARCH)
RUN_LR = LEARNING_RATE
RUN_HIDDEN_DIM = HIDDEN_DIM
RUN_DROPOUT = DROPOUT

In [ ]:
# --- Derived paths (do not edit below) ---
_prefix = "pa" if DATA == "pituitary" else "nph"
_base = Path("timelines")

if HORIZON is None:
    TREATMENT_DIR = _base / f"{_prefix}_treatment_timelines_clean"
    CONTROL_DIR   = _base / f"{_prefix}_control_timelines_clean"
else:
    TREATMENT_DIR = _base / f"{_prefix}_treatment_timelines_clean_h{HORIZON}"
    CONTROL_DIR   = _base / f"{_prefix}_control_timelines_clean_h{HORIZON}"

RESULTS_FILE = f"{_prefix}_sequence_model_experiment_results.csv"

# --- Sanity print ---
print("TREATMENT_DIR:", TREATMENT_DIR, "exists:", TREATMENT_DIR.exists())
print("CONTROL_DIR  :", CONTROL_DIR, "exists:", CONTROL_DIR.exists())
print("RESULTS_FILE:", RESULTS_FILE)

Pre-Processing (same as BoW Notebook)

In [ ]:
# Load treatment and control data
treatment_data = load_and_extract_entities(TREATMENT_DIR, 1)
control_data = load_and_extract_entities(CONTROL_DIR, 0)

# Combine into a single list
all_patient_data = treatment_data + control_data

print(f"\nLoaded {len(treatment_data)} treatment patients and {len(control_data)} control patients.")
print(f"Total patients: {len(all_patient_data)}")

In [ ]:
# Prepare DataFrame
agg_df = pd.DataFrame(all_patient_data)

# Shuffle the DataFrame to ensure patient order is random
agg_df = agg_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# Display summary
print(f"Total patients: {len(agg_df)}")
print(f"\nLabel distribution:")
print(agg_df['label'].value_counts())
print(f"\nEntity count summary:")
print(agg_df['num_entities'].describe())
display(agg_df.head())

In [ ]:
# Data Splitting
train_df, test_df = train_test_split(
    agg_df,
    test_size=0.2,
    stratify=agg_df["label"],
    random_state=RANDOM_STATE
)

print("Data split into training and testing sets:")
print(f"Training set size: {len(train_df)}")
print(f"Test set size    : {len(test_df)}")

print("\nLabel distribution in training set:")
print(train_df['label'].value_counts(normalize=True).round(3))

print("\nLabel distribution in test set:")
print(test_df['label'].value_counts(normalize=True).round(3))

Pre-Processing

In [ ]:
# Oversample the Minority Class in the Training Data

# Isolate the minority and majority classes
train_majority = train_df[train_df['label'] == 0]
train_minority = train_df[train_df['label'] == 1]

# Oversample the minority class to match the majority class
train_minority_oversampled = train_minority.sample(
    n=len(train_majority), 
    replace=True, 
    random_state=RANDOM_STATE
)

# Combine the original majority with the oversampled minority
train_df_balanced = pd.concat([train_majority, train_minority_oversampled])

# Shuffle the new balanced dataframe
train_df_balanced = train_df_balanced.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Original training label distribution:")
print(train_df['label'].value_counts())
print("\nBalanced training label distribution after oversampling:")
print(train_df_balanced['label'].value_counts())

In [ ]:
# Build Vocabulary from Training Data
word_counts = Counter()
# Use the balanced training set to build the vocabulary
for entities in train_df_balanced['entities']:
    word_counts.update(entities)

# Create a vocabulary mapping each word to an index
vocab = {word: i+2 for i, word in enumerate(word_counts)}
vocab['<pad>'] = 0
vocab['<unk>'] = 1

# Create the inverse mapping
idx_to_vocab = {i: word for word, i in vocab.items()}

# Create vocab size var
VOCAB_SIZE = len(vocab)
print(f"Vocabulary size (including special tokens): {VOCAB_SIZE}")

In [ ]:
# Analyze the distribution of sequence lengths
# Use the balanced training set for this analysis
sequence_lengths = train_df_balanced['entities'].apply(len)

# Plot the distribution
plt.figure(figsize=(10, 6))
sns.histplot(sequence_lengths, bins=50, kde=True)
plt.title('Distribution of Sequence Lengths in Balanced Training Data')
plt.xlabel('Sequence Length')
plt.ylabel('Frequency')

# Calculate and display key percentiles
percentile_95 = int(sequence_lengths.quantile(0.95))
print(f"Description of sequence lengths:")
print(sequence_lengths.describe())
print(f"\n95th percentile: {percentile_95}")
plt.axvline(percentile_95, color='r', linestyle='--', label=f'95th Percentile ({percentile_95})')
plt.legend()
plt.show()

In [ ]:
# Set the maximum sequence length based on the 95th percentile
MAX_SEQ_LENGTH = percentile_95
print(f"Set MAX_SEQ_LENGTH to: {MAX_SEQ_LENGTH}")

In [ ]:
# Apply the encoding / padding function to the training and test sets
train_df_balanced['encoded'] = train_df_balanced['entities'].apply(
    lambda x: encode_and_pad(x, vocab, MAX_SEQ_LENGTH)
)
test_df['encoded'] = test_df['entities'].apply(
    lambda x: encode_and_pad(x, vocab, MAX_SEQ_LENGTH)
)

print("\nExample of an encoded and padded sequence:")
display(train_df_balanced[['entities', 'encoded', 'label']].head())

In [ ]:
# Create a PyTorch Dataset
class PatientDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Ensure the sequence is a list of integers before converting to tensor
        sequence = self.sequences.iloc[idx]
        label = self.labels.iloc[idx]
        
        return {
            'sequence': torch.tensor(sequence, dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.float)
        }

In [ ]:
# Create Datasets for training and testing
train_dataset = PatientDataset(train_df_balanced['encoded'], train_df_balanced['label'])
test_dataset = PatientDataset(test_df['encoded'], test_df['label'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders created with batch size: {BATCH_SIZE}")

Model Training (Fixed Hyperparams)

In [ ]:
# Model Selection (MODEL_NAME is set in the config cell)

print(f"Available models: {list(MODELS.keys())}")
print(f"Selected model: {MODEL_NAME}")

In [ ]:
# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Instantiate the model (uses RUN_* from config or from HP search)
model = get_model(
    model_name=MODEL_NAME,
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_dim=RUN_HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    dropout_rate=RUN_DROPOUT
).to(device)

print(f"Model architecture: {MODEL_NAME}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

print("\nHyperparameter config (this run):")
print(f"  RUN_LR         = {RUN_LR}")
print(f"  RUN_HIDDEN_DIM = {RUN_HIDDEN_DIM}")
print(f"  RUN_DROPOUT    = {RUN_DROPOUT}")
print(f"  EMBED_DIM      = {EMBED_DIM}")
print(f"  NUM_EPOCHS     = {NUM_EPOCHS}")
print(f"  BATCH_SIZE     = {BATCH_SIZE}")

In [ ]:
# Loss function
criterion = nn.BCEWithLogitsLoss()

In [ ]:
# Optimizer (uses RUN_LR from config or from HP search)
optimizer = torch.optim.Adam(model.parameters(), lr=RUN_LR)

In [ ]:
# Training Loop with Early Stopping
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

# Early stopping parameters
best_test_loss = float('inf')
patience = EARLY_STOPPING_PATIENCE
patience_counter = 0
best_model_state = None
min_delta = EARLY_STOPPING_MIN_DELTA

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    
    # We can also check training accuracy if needed
    _, train_acc = evaluate(model, train_loader, criterion, device)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    
    # Early stopping logic with tolerance
    if test_loss < (best_test_loss - min_delta):  # Only count as improvement if significant
        best_test_loss = test_loss
        patience_counter = 0
        # Save the best model state
        best_model_state = model.state_dict().copy()
        print(f'Epoch {epoch+1:02}/{NUM_EPOCHS} | '
              f'Train Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}% | '
              f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}% | '
              f'✓ Best model (patience: {patience_counter}/{patience})')
    else:
        patience_counter += 1
        print(f'Epoch {epoch+1:02}/{NUM_EPOCHS} | '
              f'Train Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}% | '
              f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}% | '
              f'No improvement (patience: {patience_counter}/{patience})')
        
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            print(f"Best test loss was {best_test_loss:.4f}")
            # Restore best model
            model.load_state_dict(best_model_state)
            break

# If we completed all epochs, restore best model anyway
if patience_counter < patience and best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nTraining completed. Best test loss: {best_test_loss:.4f}")

In [ ]:
# Plotting the training and validation curves
plot_training_curves(train_losses, test_losses, train_accuracies, test_accuracies)

In [ ]:
# Get predictions on the test set
y_test_labels, y_test_preds, y_test_probs = get_predictions(model, test_loader, device)

In [ ]:
# Metrics and Confusion Matrix
evaluate_model(y_test_labels, y_test_probs, threshold=0.5)

Threshold Tuning

In [ ]:
# Threshold Tuning
best_threshold, best_f1 = find_optimal_threshold(y_test_labels, y_test_probs, plot=True)

In [ ]:
# Evaluate Model with the New Threshold
evaluate_model(y_test_labels, y_test_probs, threshold=best_threshold, 
               title_suffix=" – Using Optimal Threshold")

Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning (learning rate × hidden dimension × dropout)
if USE_HP_SEARCH:
    search_results = []

    print("=" * 60)
    print("Hyperparameter search: hidden_dim × learning rate × dropout")
    print("=" * 60)

    for lr in LEARNING_RATES:
        for hidden_dim in HIDDEN_DIMS:
            for dropout_rate in DROPOUT_RATES:
                print(f"\n--- Testing lr={lr:.4f}, hidden_dim={hidden_dim}, dropout={dropout_rate:.1f} ---")
                
                # new model for each combination
                model_search = get_model(MODEL_NAME, VOCAB_SIZE, EMBED_DIM, hidden_dim, OUTPUT_DIM, dropout_rate).to(device)
                criterion = nn.BCEWithLogitsLoss()
                optimizer = torch.optim.Adam(model_search.parameters(), lr=lr)
                
                # Early stopping for hyperparameter search (same parameters as main training)
                best_search_loss = float('inf')
                search_patience = EARLY_STOPPING_PATIENCE
                search_patience_counter = 0
                search_min_delta = EARLY_STOPPING_MIN_DELTA
                best_search_model_state = None
                
                for epoch in range(NUM_EPOCHS_SEARCH):
                    train_loss = train_epoch(model_search, train_loader, optimizer, criterion, device)
                    test_loss, _ = evaluate(model_search, test_loader, criterion, device)
                    
                    if test_loss < (best_search_loss - search_min_delta):
                        best_search_loss = test_loss
                        search_patience_counter = 0
                        best_search_model_state = model_search.state_dict().copy()
                    else:
                        search_patience_counter += 1
                        if search_patience_counter >= search_patience:
                            # Restore best model before evaluation
                            model_search.load_state_dict(best_search_model_state)
                            break
                
                # Restore best model if we completed all epochs
                if search_patience_counter < search_patience and best_search_model_state is not None:
                    model_search.load_state_dict(best_search_model_state)
                
                # Evaluation pass - use get_predictions utility
                y_search_labels, _, y_search_probs = get_predictions(model_search, test_loader, device)
                
                # Use utility function for threshold finding (without plotting)
                best_threshold_search, best_f1 = find_optimal_threshold(y_search_labels, y_search_probs, plot=False)
                roc_auc = roc_auc_score(y_search_labels, y_search_probs)
                
                search_results.append({
                    'learning_rate': lr,
                    'hidden_dim': hidden_dim,
                    'dropout_rate': dropout_rate,
                    'f1_score': best_f1,
                    'roc_auc': roc_auc,
                    'best_threshold': best_threshold_search
                })
                
                print(f"  F1={best_f1:.3f} | ROC-AUC={roc_auc:.3f} | threshold={best_threshold_search:.4f}")

    results_df = pd.DataFrame(search_results).sort_values('f1_score', ascending=False)
    display(results_df)

    print("\nBest configuration:")
    best_cfg = results_df.iloc[0]
    print(f"  lr={best_cfg.learning_rate:.4f}, hidden_dim={int(best_cfg.hidden_dim)}, dropout={best_cfg.dropout_rate:.1f}")
    print(f"  F1={best_cfg.f1_score:.3f}, ROC-AUC={best_cfg.roc_auc:.3f}")

    best_search_threshold = best_cfg.best_threshold
    print(f"  Suggested threshold={best_search_threshold:.4f}")
else:
    print("Skipping hyperparameter search (USE_HP_SEARCH=False).")


In [ ]:
# Capture best hyperparameters from the search table
if USE_HP_SEARCH:
    BEST_LR = results_df.iloc[0]['learning_rate']
    BEST_HIDDEN_DIM = int(results_df.iloc[0]['hidden_dim'])
    BEST_DROPOUT = results_df.iloc[0]['dropout_rate']
    BEST_SEARCH_THRESHOLD = results_df.iloc[0]['best_threshold']

    RUN_LR = BEST_LR
    RUN_HIDDEN_DIM = BEST_HIDDEN_DIM
    RUN_DROPOUT = BEST_DROPOUT

    print(f"Using best hyperparameters → lr={RUN_LR:.4f}, hidden_dim={RUN_HIDDEN_DIM}, dropout={RUN_DROPOUT:.1f}")
else:
    print("Using config hyperparameters (no search).")

In [ ]:
# Final training with best hyperparameters (BEST_LR, BEST_HIDDEN_DIM, BEST_DROPOUT from search cell)
if USE_HP_SEARCH:
    # Final training with best hyperparameters (RUN_* set by capture-best cell)
    model = get_model(
        MODEL_NAME,
        VOCAB_SIZE,
        EMBED_DIM,
        RUN_HIDDEN_DIM,
        OUTPUT_DIM,
        RUN_DROPOUT,
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=RUN_LR)

    train_losses = []
    test_losses = []
    train_accuracies = []
    test_accuracies = []

    best_test_loss = float('inf')
    patience = EARLY_STOPPING_PATIENCE
    patience_counter = 0
    best_model_state = None
    min_delta = EARLY_STOPPING_MIN_DELTA

    for epoch in range(NUM_EPOCHS):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        _, train_acc = evaluate(model, train_loader, criterion, device)

        train_losses.append(train_loss)
        test_losses.append(test_loss)
        train_accuracies.append(train_acc)
        test_accuracies.append(test_acc)

        if test_loss < (best_test_loss - min_delta):
            best_test_loss = test_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
            print(
                f'Epoch {epoch+1:02}/{NUM_EPOCHS} | '
                f'Train Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}% | '
                f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}% | '
                f'✓ Best model (patience: {patience_counter}/{patience})'
            )
        else:
            patience_counter += 1
            print(
                f'Epoch {epoch+1:02}/{NUM_EPOCHS} | '
                f'Train Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}% | '
                f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}% | '
                f'No improvement (patience: {patience_counter}/{patience})'
            )

            if patience_counter >= patience:
                print(f"\nEarly stopping triggered at epoch {epoch+1}")
                print(f"Best test loss was {best_test_loss:.4f}")
                model.load_state_dict(best_model_state)
                break

    if patience_counter < patience and best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nTraining completed. Best test loss: {best_test_loss:.4f}")
else:
    print("Skipping final training (USE_HP_SEARCH=False). Using model from first training loop.")

In [ ]:
# Loss and accuracy plots for tuned model
plot_training_curves(train_losses, test_losses, train_accuracies, test_accuracies, 
                     title_suffix=" (Tuned Model)")

In [ ]:
# Baseline evaluation with threshold=0.5
y_test_labels, _, y_test_probs = get_predictions(model, test_loader, device)
evaluate_model(y_test_labels, y_test_probs, threshold=0.5, 
               title_suffix=" (Tuned Model, threshold=0.5)")

In [ ]:
# Threshold tuning for tuned model
best_threshold, best_f1 = find_optimal_threshold(y_test_labels, y_test_probs, plot=True)

In [ ]:
# Final evaluation using the tuned threshold
evaluate_model(y_test_labels, y_test_probs, threshold=best_threshold, 
               title_suffix=" (Tuned Model, Optimal Threshold)")

In [ ]:
# Save Experiment Results to CSV

# Get class distributions
# Training set (what the model actually saw - after oversampling)
train_class_counts = train_df_balanced['label'].value_counts().sort_index()
train_class_0 = train_class_counts.get(0, 0)
train_class_1 = train_class_counts.get(1, 0)
train_total = len(train_df_balanced)
train_class_balance_ratio = round(train_class_0 / train_class_1, 4) if train_class_1 > 0 else None
train_class_0_pct = round(train_class_0 / train_total * 100, 2) if train_total > 0 else 0
train_class_1_pct = round(train_class_1 / train_total * 100, 2) if train_total > 0 else 0

# Test set
test_class_counts = test_df['label'].value_counts().sort_index()
test_class_0 = test_class_counts.get(0, 0)
test_class_1 = test_class_counts.get(1, 0)
test_total = len(test_df)
test_class_balance_ratio = round(test_class_0 / test_class_1, 4) if test_class_1 > 0 else None
test_class_0_pct = round(test_class_0 / test_total * 100, 2) if test_total > 0 else 0
test_class_1_pct = round(test_class_1 / test_total * 100, 2) if test_total > 0 else 0

# Get metrics with default threshold (0.5)
y_test_labels_default, y_test_preds_default, y_test_probs_default = get_predictions(model, test_loader, device)
acc_default = accuracy_score(y_test_labels_default, y_test_preds_default)
prec_default = precision_score(y_test_labels_default, y_test_preds_default)
rec_default = recall_score(y_test_labels_default, y_test_preds_default)
f1_default = f1_score(y_test_labels_default, y_test_preds_default)
roc_auc_default = roc_auc_score(y_test_labels_default, y_test_probs_default)
cm_default = confusion_matrix(y_test_labels_default, y_test_preds_default)

# Get metrics with optimal threshold
y_test_labels_opt, _, y_test_probs_opt = get_predictions(model, test_loader, device)
best_threshold_opt, best_f1_opt = find_optimal_threshold(y_test_labels_opt, y_test_probs_opt, plot=False)
y_pred_opt = (y_test_probs_opt >= best_threshold_opt).astype(int)
acc_opt = accuracy_score(y_test_labels_opt, y_pred_opt)
prec_opt = precision_score(y_test_labels_opt, y_pred_opt)
rec_opt = recall_score(y_test_labels_opt, y_pred_opt)
f1_opt = f1_score(y_test_labels_opt, y_pred_opt)
roc_auc_opt = roc_auc_score(y_test_labels_opt, y_test_probs_opt)
cm_opt = confusion_matrix(y_test_labels_opt, y_pred_opt)

# Collect all experiment information
experiment_data = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'data': DATA,
    'horizon_days': HORIZON,
    'model_architecture': MODEL_NAME,
    'embed_dim': EMBED_DIM,
    'hidden_dim': RUN_HIDDEN_DIM,
    'dropout_rate': RUN_DROPOUT,
    'learning_rate': RUN_LR,
    'batch_size': BATCH_SIZE,
    'max_seq_length': MAX_SEQ_LENGTH,
    'vocab_size': VOCAB_SIZE,
    'num_epochs': NUM_EPOCHS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'early_stopping_min_delta': EARLY_STOPPING_MIN_DELTA,

    # Test set metrics with DEFAULT threshold (0.5)
    'test_accuracy_default': round(acc_default, 4),
    'test_precision_default': round(prec_default, 4),
    'test_recall_default': round(rec_default, 4),
    'test_f1_default': round(f1_default, 4),
    'test_roc_auc_default': round(roc_auc_default, 4),

    # Confusion matrix with default threshold (as string)
    'confusion_matrix_default': str(cm_default.tolist()),

    # Threshold tuning results
    'optimal_threshold': round(best_threshold_opt, 4),
    'f1_at_optimal_threshold': round(best_f1_opt, 4),

    # Test set metrics with OPTIMAL threshold
    'test_accuracy_optimal': round(acc_opt, 4),
    'test_precision_optimal': round(prec_opt, 4),
    'test_recall_optimal': round(rec_opt, 4),
    'test_f1_optimal': round(f1_opt, 4),
    'test_roc_auc_optimal': round(roc_auc_opt, 4),

    # Confusion matrix with optimal threshold (as string)
    'confusion_matrix_optimal': str(cm_opt.tolist()),

    # Training set class distribution (after oversampling)
    'train_total': train_total,
    'train_class_0': train_class_0,
    'train_class_1': train_class_1,
    'train_class_0_pct': train_class_0_pct,
    'train_class_1_pct': train_class_1_pct,
    'train_class_balance_ratio': train_class_balance_ratio,

    # Test set class distribution
    'test_total': test_total,
    'test_class_0': test_class_0,
    'test_class_1': test_class_1,
    'test_class_0_pct': test_class_0_pct,
    'test_class_1_pct': test_class_1_pct,
    'test_class_balance_ratio': test_class_balance_ratio,

    # Data info
    'n_train_original': len(train_df),          # before oversampling
    'n_train_balanced': len(train_df_balanced), # after oversampling
    'n_test': len(test_df),
    'n_features': VOCAB_SIZE,                   # vocabulary size
}

# Define CSV column headers (in order)
csv_headers = [
    'timestamp',
    'data',
    'horizon_days',
    'model_architecture',
    'embed_dim',
    'hidden_dim',
    'dropout_rate',
    'learning_rate',
    'batch_size',
    'max_seq_length',
    'vocab_size',
    'num_epochs',
    'early_stopping_patience',
    'early_stopping_min_delta',
    'test_accuracy_default',
    'test_precision_default',
    'test_recall_default',
    'test_f1_default',
    'test_roc_auc_default',
    'confusion_matrix_default',
    'optimal_threshold',
    'f1_at_optimal_threshold',
    'test_accuracy_optimal',
    'test_precision_optimal',
    'test_recall_optimal',
    'test_f1_optimal',
    'test_roc_auc_optimal',
    'confusion_matrix_optimal',
    'train_total',
    'train_class_0',
    'train_class_1',
    'train_class_0_pct',
    'train_class_1_pct',
    'train_class_balance_ratio',
    'test_total',
    'test_class_0',
    'test_class_1',
    'test_class_0_pct',
    'test_class_1_pct',
    'test_class_balance_ratio',
    'n_train_original',
    'n_train_balanced',
    'n_test',
    'n_features',
]

# Check if file exists to determine if we need to write headers
file_exists = os.path.exists(RESULTS_FILE)

# Append row to CSV
with open(RESULTS_FILE, 'a', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=csv_headers)

    # Write header if file is new
    if not file_exists:
        writer.writeheader()
        print(f"Created new results file: {RESULTS_FILE}")

    # Write experiment data
    writer.writerow(experiment_data)
    print(f"\n{'='*50}")
    print("EXPERIMENT RESULTS SAVED")
    print(f"{'='*50}")
    print(f"File: {RESULTS_FILE}")
    print(f"Data: {DATA}")
    print(f"Horizon (days): {HORIZON}")
    print(f"Model Architecture: {MODEL_NAME}")
    print(f"\nTraining Set Class Distribution (after oversampling):")
    print(f"  Class 0: {train_class_0} ({train_class_0_pct}%)")
    print(f"  Class 1: {train_class_1} ({train_class_1_pct}%)")
    print(f"  Balance Ratio (0:1): {train_class_balance_ratio}")
    print(f"\nTest Set Class Distribution:")
    print(f"  Class 0: {test_class_0} ({test_class_0_pct}%)")
    print(f"  Class 1: {test_class_1} ({test_class_1_pct}%)")
    print(f"  Balance Ratio (0:1): {test_class_balance_ratio}")
    print(f"\nTest F1 (optimal threshold): {f1_opt:.4f}")
    print(f"Test ROC-AUC (optimal threshold): {roc_auc_opt:.4f}")
    print(f"{'='*50}\n")

Inference

In [ ]:
# Inference for sequence model.
# Requires (from training cells): model, vocab, MAX_SEQ_LENGTH, device, PatientDataset, BATCH_SIZE, best_threshold

import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from utils import encode_and_pad

required_vars = ["model", "vocab", "MAX_SEQ_LENGTH", "device", "PatientDataset"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise NameError(
        f"Missing variables: {missing_vars}. "
        "Run training cells first so model/vocab/MAX_SEQ_LENGTH/device/PatientDataset exist."
    )


def _extract_entities_from_timeline_file(timeline_file: Path):
    """Return (patient_id, entities_list) with the SAME normalization as training
    (lowercase + strip). If this drifts, vocab coverage collapses -> flat predictions."""
    with open(timeline_file, "r", encoding="utf-8") as f:
        timeline = json.load(f)
    patient_id = str(timeline.get("patient_id", Path(timeline_file).stem))
    events = timeline.get("events", [])
    entities = [
        e["entity_preferred_name"].lower().strip()
        for e in events
        if e.get("entity_preferred_name")
    ]
    return patient_id, entities


def _expand_input_paths(paths):
    if isinstance(paths, (str, Path)):
        paths = [paths]
    all_files = []
    for p in paths:
        p = Path(p)
        if p.is_dir():
            all_files.extend(sorted(p.glob("*.json")))
        elif p.is_file():
            all_files.append(p)
        else:
            raise FileNotFoundError(f"Path does not exist: {p}")
    if not all_files:
        raise ValueError("No timeline JSON files found in provided path(s).")
    return all_files


def _resolve_threshold(passed):
    """Priority: passed arg -> INFERENCE_THRESHOLD -> best_threshold. No silent fallback to 0.5."""
    g = globals()
    if passed is not None:
        return float(passed), "argument"
    if "INFERENCE_THRESHOLD" in g and g["INFERENCE_THRESHOLD"] is not None:
        return float(g["INFERENCE_THRESHOLD"]), "INFERENCE_THRESHOLD"
    if "best_threshold" in g and g["best_threshold"] is not None:
        return float(g["best_threshold"]), "best_threshold (Threshold Tuning cell)"
    raise RuntimeError(
        "No inference threshold available. Pass threshold=..., set INFERENCE_THRESHOLD, "
        "or run the Threshold Tuning cell first."
    )


def predict_timelines(paths, threshold=None, batch_size=None, verbose=True):
    """Run inference on one or many timeline JSON files / directories.
    Returns a DataFrame with patient_id, num_entities, probability, pred_label."""
    files = _expand_input_paths(paths)

    patient_ids, entity_lists, raw_counts = [], [], []
    for f in files:
        pid, entities = _extract_entities_from_timeline_file(f)
        patient_ids.append(pid)
        entity_lists.append(entities)
        raw_counts.append(len(entities))

    # Vocab coverage diagnostic
    total_tokens = sum(len(el) for el in entity_lists)
    known_tokens = sum(1 for el in entity_lists for e in el if e in vocab)
    coverage = (100.0 * known_tokens / total_tokens) if total_tokens else 0.0
    if verbose:
        print(f"Vocab coverage: {known_tokens}/{total_tokens} tokens ({coverage:.2f}%)")
        if coverage < 5.0:
            print("WARNING: very low vocab coverage - predictions may be near-constant.")

    # Encode + pad using the training vocab and MAX_SEQ_LENGTH
    encoded = [encode_and_pad(el, vocab, MAX_SEQ_LENGTH) for el in entity_lists]

    # Dataset/DataLoader using the same PatientDataset (dummy labels for inference)
    #ds = PatientDataset(encoded, [0] * len(encoded))
    ds = PatientDataset(pd.Series(encoded), pd.Series([0] * len(encoded)))
    bs = batch_size if batch_size is not None else globals().get("BATCH_SIZE", 32)
    loader = DataLoader(ds, batch_size=bs, shuffle=False)

    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            seq = batch["sequence"].to(device)
            logits = model(seq).squeeze(1)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs.tolist())
    all_probs = np.array(all_probs, dtype=float)

    threshold_value, threshold_source = _resolve_threshold(threshold)
    if verbose:
        print(f"Using threshold={threshold_value:.4f} (source: {threshold_source})")

    preds = (all_probs >= threshold_value).astype(int)

    results = pd.DataFrame({
        "timeline_path": [str(f) for f in files],
        "patient_id": patient_ids,
        "num_entities": raw_counts,
        "prob_treatment_class_1": all_probs,
        "pred_label": preds,
        "threshold_used": threshold_value,
        "threshold_source": threshold_source,
    }).sort_values("prob_treatment_class_1", ascending=False).reset_index(drop=True)

    return results

In [ ]:
# Inference over entire directory (choose the horizon that matches training)
inference_dir = Path("timelines/ed_timelines_clean_hfirstcontact")
pred_dir = predict_timelines(inference_dir)
print(f"Predictions generated: {len(pred_dir)}")
display(pred_dir.head(20))

In [ ]:
# Save inference outputs
out_csv = Path("nph_sequence_ed_inference_predictions.csv")
pred_dir.to_csv(out_csv, index=False)
print(f"Saved: {out_csv.resolve()}")

In [ ]:
# Inference diagnostics on pred_dir (sequence model)

import numpy as np
import pandas as pd

if "pred_dir" not in globals():
    raise NameError("pred_dir not found. Run inference first (e.g. pred_dir = predict_timelines(...)).")

print("=" * 70)
print("INFERENCE PREDICTION SUMMARY")
print("=" * 70)

n = len(pred_dir)
print(f"Total inferred timelines: {n}")

pred_counts = pred_dir["pred_label"].value_counts().sort_index()
pred_props = pred_dir["pred_label"].value_counts(normalize=True).sort_index()

n_control = int(pred_counts.get(0, 0))
n_treat = int(pred_counts.get(1, 0))
p_control = float(pred_props.get(0, 0.0))
p_treat = float(pred_props.get(1, 0.0))

print("\nPredicted class distribution:")
print(f"  Pred 0 (control)   : {n_control:>6}  ({p_control:>7.2%})")
print(f"  Pred 1 (treatment) : {n_treat:>6}  ({p_treat:>7.2%})")

probs = pred_dir["prob_treatment_class_1"].astype(float)
print("\nProbability summary (prob_treatment_class_1):")
print(probs.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

print("\nProbability buckets:")
bins = [-np.inf, 0.2, 0.4, 0.6, 0.8, np.inf]
labels = ["<=0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", ">0.8"]
bucketed = pd.cut(probs, bins=bins, labels=labels)
print(bucketed.value_counts().sort_index())

print("\nMost confident predicted treatment (top 10):")
display(pred_dir.sort_values("prob_treatment_class_1", ascending=False).head(10))

print("\nMost confident predicted control (top 10):")
display(pred_dir.sort_values("prob_treatment_class_1", ascending=True).head(10))